# Step 1: Load Data

In [114]:
import pandas as pd

df = pd.read_csv('../Data/flipkart_product.csv', encoding='latin1')
df.head()

,ProductName,Price,Rate,Review,Summary
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Super!,Great cooler.. excellent air flow and for this...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Awesome,Best budget 2 fit cooler. Nice cooling
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,The quality is good but the power of air is de...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",1,Useless product,Very bad product it's a only a fan
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,Ok ok product


# Step 2: Data Cleaning

In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189874 entries, 0 to 189873
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   ProductName  189874 non-null  object
 1   Price        189873 non-null  object
 2   Rate         189873 non-null  object
 3   Review       189870 non-null  object
 4   Summary      189860 non-null  object
dtypes: object(5)
memory usage: 7.2+ MB


In [119]:
df.isnull().sum()

ProductName     0
Price           1
Rate            1
Review          4
Summary        14
dtype: int64

In [121]:
df['Rate'].value_counts()

Rate
5                                                              108694
4                                                               39653
1                                                               19607
3                                                               15681
2                                                                6234
Pigeon Favourite Electric Kettle?ÿ?ÿ(1.5 L, Silver, Black)          1
Bajaj DX 2 L/W Dry Iron                                             1
Nova Plus Amaze NI 10 1100 W Dry Iron?ÿ?ÿ(Grey & Turquoise)         1
s                                                                   1
Name: count, dtype: int64

In [123]:
df['Rate'].dtype
df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')

# Step 3: Sentiment Creation

In [126]:
def convert_sentiment(rate):
    if rate>=4:
        return "Positive"
    elif rate==3:
        return "Neutral"
    else:
        return "Negative"
df['Sentiment']=df['Rate'].apply(convert_sentiment)


In [128]:
df['Sentiment'].value_counts()

Sentiment
Positive    148347
Negative     25846
Neutral      15681
Name: count, dtype: int64

In [130]:
df['Sentiment'].value_counts(normalize=True) * 100

Sentiment
Positive    78.129180
Negative    13.612185
Neutral      8.258635
Name: proportion, dtype: float64

In [132]:
df.to_csv('../data/cleaned_reviews.csv', index=False)

# Step 4: Text Preprocessing

In [134]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove punctuation & special characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # 3. Tokenization
    words = word_tokenize(text)
    
    # 4. Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # 5. Join back
    return " ".join(words)

# Apply on Review column
df['Review'] = df['Review'].astype(str)
df['clean_text'] = df['Review'].apply(clean_text)

df[['Review', 'clean_text']].head()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Review,clean_text
0,Super!,super
1,Awesome,awesome
2,Fair,fair
3,Useless product,useless product
4,Fair,fair


# Step 5: Feature Engineering (TF-IDF)

In [181]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['clean_text'])

In [182]:
print(X.shape)

(189874, 1054)


In [183]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df['Sentiment'], test_size=0.2, random_state=42
)

# Step 6: Model Building

In [185]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)


In [186]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred_rf=rf.predict(X_test)


In [187]:
from sklearn.metrics import accuracy_score

print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("Random Forest:", accuracy_score(y_test, y_pred_rf))

Logistic Regression: 0.9507570770243581
Random Forest: 0.9508360763660303


In [188]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_lr)
print(cm)

[[ 4180     6   949]
 [    6  2189   886]
 [    7    16 29736]]


In [189]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

    Negative       1.00      0.81      0.90      5135
     Neutral       0.99      0.71      0.83      3081
    Positive       0.94      1.00      0.97     29759

    accuracy                           0.95     37975
   macro avg       0.98      0.84      0.90     37975
weighted avg       0.95      0.95      0.95     37975



In [190]:
confusion_matrix(y_test, y_pred_lr)
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

    Negative       1.00      0.81      0.90      5135
     Neutral       0.99      0.71      0.83      3081
    Positive       0.94      1.00      0.97     29759

    accuracy                           0.95     37975
   macro avg       0.98      0.84      0.90     37975
weighted avg       0.95      0.95      0.95     37975



In [191]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=0.5, max_iter=200)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [192]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)

# Step 7: Evaluation

In [194]:

tfidf = TfidfVectorizer(ngram_range=(1,2))  # unigram + bigram
X = tfidf.fit_transform(df['clean_text'])
X_train, X_test, y_train, y_test = train_test_split(
    X, df['Sentiment'], test_size=0.2, random_state=42
)

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print(classification_report(y_test, y_pred))
print(X.shape)

              precision    recall  f1-score   support

    Negative       1.00      0.81      0.90      5135
     Neutral       0.99      0.71      0.83      3081
    Positive       0.94      1.00      0.97     29759

    accuracy                           0.95     37975
   macro avg       0.98      0.84      0.90     37975
weighted avg       0.95      0.95      0.95     37975

(189874, 2944)


In [195]:
print(tfidf.get_feature_names_out()[:20])

['absolute' 'absolute beast' 'absolute rubbish' 'ac' 'ac cooler'
 'acceptable' 'achcha' 'achcha hai' 'act' 'act lying' 'advisable'
 'advisable buy' 'affordable' 'affordable cost' 'affordable price'
 'affordable well' 'aftersale' 'aftersale service' 'agaro' 'ah']


In [196]:
print(X[0])

  (0, 2530)	1.0


# Step 8: Model Improvement

In this step, we improve the model by:
- Using n-grams in TF-IDF
- Tuning model parameters
- Comparing performance before and after improvement

Code — 1 Apply n-grams

In [199]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,2))  # unigram + bigram
X = tfidf.fit_transform(df['clean_text'])

Code — 2 Train-Test Split AGAIN

In [201]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df['Sentiment'], test_size=0.2, random_state=42
)

Code — 3 Train Improved Model

In [207]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=0.5, max_iter=200)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

Code — 4 Evaluate AGAIN

In [209]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       1.00      0.81      0.90      5135
     Neutral       0.99      0.71      0.83      3081
    Positive       0.94      1.00      0.97     29759

    accuracy                           0.95     37975
   macro avg       0.98      0.84      0.90     37975
weighted avg       0.95      0.95      0.95     37975

